### Desafio - Aula 6.5 (Métricas de Classificação)

In [20]:
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score

In [23]:
def divide(num, den):
    return num / den if den != 0 else 0.0

def accuracy(tp, tn, fp, fn):
    return divide(tp + tn, tp + tn + fp + fn)

def precision(tp, fp):
    return divide(tp, tp + fp)

def recall(tp, fn):
    return divide(tp, tp + fn)

def f1_score(p, r):
    return divide(2 * p * r, p + r)

def contar(y_real, y_pred):
    tp = np.sum((y_real == 1) & (y_pred == 1))
    tn = np.sum((y_real == 0) & (y_pred == 0))
    fp = np.sum((y_real == 0) & (y_pred == 1))
    fn = np.sum((y_real == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def matriz_confusao(y_real, y_pred):
    tp, tn, fp, fn = contar(y_real, y_pred)
    return np.array([[tn, fp],
                     [fn, tp]])

def meu_classification_report(y_real, y_pred):
    classes = np.unique(np.concatenate([y_real, y_pred]))
    n = len(classes)
    M = matriz_confusao(y_real, y_pred)
    
    if M is None:
        raise ValueError("A matriz_confusao retornou None. Este código foi feito para suportar estritamente as classes binárias {0, 1}.")
    
    # Extraindo os valores verdadeiros/falsos da matriz binária para a função accuracy
    tn_val, fp_val = M[0, 0], M[0, 1]
    fn_val, tp_val = M[1, 0], M[1, 1]
    acc = accuracy(tp_val, tn_val, fp_val, fn_val)
    
    tp = np.diag(M)
    fp = M.sum(axis=0) - tp
    fn = M.sum(axis=1) - tp
    support = M.sum(axis=1)
    
    # Evitar erro de Numpy:
    p = np.array([precision(tp[i], fp[i]) for i in range(n)])
    r  = np.array([recall(tp[i], fn[i]) for i in range(n)])
    f1   = np.array([f1_score(p[i], r[i]) for i in range(n)])

    # Médias aritméticas    
    macro_p = p.mean()
    macro_r = r.mean()
    macro_f1 = f1.mean()
    
    tot = support.sum()
    if tot > 0: # Médias harmônicas
        w_p = np.average(p, weights=support)
        w_r = np.average(r, weights=support)
        w_f1 = np.average(f1, weights=support)
    else:
        w_p = w_r = w_f1 = 0.0

    # Assim como exercício guiado:
    
    out = f"{'':>14}{'precision':>11}{'recall':>9}{'f1-score':>10}{'support':>9}\n\n"
    for i in range(n):
        out += f"{str(classes[i]):>14}{p[i]:>11.2f}{r[i]:>9.2f}{f1[i]:>10.2f}{support[i]:>9}\n"
    out += f"\n{'accuracy':>14}{'':>20}{acc:>10.2f}{tot:>9}\n"
    out += f"{'macro avg':>14}{macro_p:>11.2f}{macro_r:>9.2f}{macro_f1:>10.2f}{tot:>9}\n"
    out += f"{'weighted avg':>14}{w_p:>11.2f}{w_r:>9.2f}{w_f1:>10.2f}{tot:>9}"
    return out

# Assim como nos slides da aula:
def curva_roc(y_real, scores):
    ordem = np.argsort(-scores, kind="mergesort")
    y, s = y_real[ordem], scores[ordem]
    fim = np.r_[np.where(np.diff(s) != 0)[0],
                len(s) - 1]
    tp = np.cumsum(y)[fim]
    fp = (fim + 1) - tp
    P, N = y.sum(), len(y) - y.sum()
    tpr = np.r_[0, tp] / P  
    fpr = np.r_[0, fp] / N
    return fpr, tpr

def auc(fpr, tpr):
    larg = fpr[1:] - fpr[:-1]
    alt = (tpr[1:] + tpr[:-1]) / 2
    return np.sum(larg * alt)     

In [25]:
# Teste
y_real = np.array([0]*12 + [1]*13)
y_pred = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
                    0, 1, 1, 1, 1,
                    1, 1, 1, 0, 1, 1, 1, 1, 1, 1])

print(meu_classification_report(y_real, y_pred))

# Teste ROC e AUC
y_bin = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])
scores = np.array([0.95, 0.10, 0.70, 0.55, 0.85, 0.38, 0.45, 0.25, 0.62, 0.78])

fpr, tpr = curva_roc(y_bin, scores)
print("\nAUC meu:", round(auc(fpr, tpr), 4))

# Conferir com sklearn
print("\nSklearn")
print(classification_report(y_real, y_pred))
print("AUC sklearn:", round(roc_auc_score(y_bin, scores), 4))


                precision   recall  f1-score  support

             0       0.90     0.75      0.82       12
             1       0.80     0.92      0.86       13

      accuracy                          0.84       25
     macro avg       0.85     0.84      0.84       25
  weighted avg       0.85     0.84      0.84       25

AUC meu: 0.84

Sklearn
              precision    recall  f1-score   support

           0       0.90      0.75      0.82        12
           1       0.80      0.92      0.86        13

    accuracy                           0.84        25
   macro avg       0.85      0.84      0.84        25
weighted avg       0.85      0.84      0.84        25

AUC sklearn: 0.84
